# Keyword Extraction v2

Improved extraction using **stemming** (SnowballStemmer) to consolidate word variants (color/colored/colorful → color),
with filtering of numbers, non-alpha tokens, and short fragments.

### Improvements over v1
- Stems instead of lemmas to merge word variants (e.g., confuse/confused/confusing → confus)
- Filters out numbers (`1`, `1/3`, `15`), non-alpha tokens (`2d/3d`, `couldnt\`see`)
- Keeps a readable representative word for each stem (the shortest word mapping to that stem)

### Input
- `figures/image_phrase_matching.csv` — produced by KeywordExtraction.ipynb

### Output
- `figures/unique_stems.csv` — all-phrases stem extraction
- `figures/unique_stems_by_topic.csv` — per-topic with image counts
- `figures/unique_stems_by_vistype.csv` — per-VisType with image counts

In [1]:
import pandas as pd
import spacy
import re
from collections import Counter, defaultdict
from nltk.stem import SnowballStemmer

nlp = spacy.load('en_core_web_sm')
stemmer = SnowballStemmer('english')

# Load image-based phrase matching (produced by KeywordExtraction.ipynb)
df_img = pd.read_csv('figures/image_phrase_matching.csv')
print(f'Loaded image_phrase_matching.csv: {len(df_img)} records, {df_img["ImageName"].nunique()} images')

topic_cols = [
    'Immediacy / Cognitive Load',
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
]
CANONICAL_VISTYPES = ['Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']

Loaded image_phrase_matching.csv: 1909 records, 493 images


In [6]:
# ── Helper functions ────────────────────────────────────────────────────────────

WHITELIST_TOKENS = {'2d', '3d', '2d/3d'}

def is_valid_token(token):
    """Check if a spaCy token is a valid content word (not stop/punct/space/number)."""
    text = token.text.lower()
    if text in WHITELIST_TOKENS:
        return True
    if token.is_stop or token.is_punct or token.is_space:
        return False
    if token.pos_ == 'NUM':
        return False
    # Must be purely alphabetic (allows hyphens inside words)
    if not re.match(r'^[a-z]+(-[a-z]+)*$', text):
        return False
    if len(text) <= 1:
        return False
    return True

def extract_stems_from_phrase(phrase):
    """Extract set of (stem, representative_word) from a single phrase."""
    if not isinstance(phrase, str) or phrase.strip() in ('', '-'):
        return set()
    doc = nlp(phrase)
    results = set()
    for token in doc:
        if not is_valid_token(token):
            continue
        word = token.lemma_.lower()
        stem = stemmer.stem(word)
        results.add((stem, word))
    return results

# Build a global stem → best representative word mapping
# Use the shortest lemma that maps to each stem
all_phrases = pd.concat([
    df_img['Original'], df_img['AIcurated'], df_img['HumanCurated']
]).dropna().unique()

stem_to_word = {}
for phrase in all_phrases:
    for stem, word in extract_stems_from_phrase(phrase):
        if stem not in stem_to_word or len(word) < len(stem_to_word[stem]):
            stem_to_word[stem] = word

def get_stems_from_phrase(phrase):
    """Extract set of stems from a phrase."""
    return {stem for stem, _ in extract_stems_from_phrase(phrase)}

print(f'Total unique stems: {len(stem_to_word)}')
# Show some examples of consolidation
examples = ['color', 'confus', 'clear', 'clutter', 'annot', 'complex', 'bright']
for s in examples:
    if s in stem_to_word:
        print(f'  stem "{s}" → "{stem_to_word[s]}"')

Total unique stems: 713
  stem "color" → "color"
  stem "confus" → "confuse"
  stem "clear" → "clear"
  stem "clutter" → "clutter"
  stem "annot" → "annotation"
  stem "complex" → "complex"
  stem "bright" → "bright"


In [7]:
# ── All-phrases unique stems with POS ──────────────────────────────────────────
def extract_stem_pos_pairs(phrases):
    """Extract unique (stem, POS) pairs from phrases, returning (representative_word, POS)."""
    stem_pos = set()
    for phrase in phrases:
        if not isinstance(phrase, str) or phrase.strip() in ('', '-'):
            continue
        doc = nlp(phrase)
        for token in doc:
            if not is_valid_token(token):
                continue
            word = token.lemma_.lower()
            stem = stemmer.stem(word)
            stem_pos.add((stem, token.pos_))
    # Convert stems to representative words
    return sorted((stem_to_word.get(s, s), pos) for s, pos in stem_pos)

cols = {'Original': 'Original', 'AIcurated': 'AIcurated', 'HumanCurated': 'HumanCurated'}
word_lists = {}
for col, label in cols.items():
    phrases = df_img[col].dropna()
    phrases = phrases[phrases.str.strip() != ''].unique()
    wp = extract_stem_pos_pairs(phrases)
    word_lists[label] = wp
    print(f'{label}: {len(phrases)} phrases → {len(wp)} unique stems')

max_len = max(len(v) for v in word_lists.values())
frames = []
for label in cols.values():
    wp = word_lists[label]
    d = pd.DataFrame(wp, columns=[f'{label}Word', f'{label}POS'])
    d = d.reindex(range(max_len)).fillna('')
    frames.append(d)

df_out = pd.concat(frames, axis=1)
csv_path = 'figures/unique_stems.csv'
df_out.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({max_len} rows)')
df_out.head(15)

Original: 1560 phrases → 829 unique stems
AIcurated: 868 phrases → 537 unique stems
HumanCurated: 397 phrases → 363 unique stems

Saved figures/unique_stems.csv  (829 rows)


,OriginalWord,OriginalPOS,AIcuratedWord,AIcuratedPOS,HumanCuratedWord,HumanCuratedPOS
0,2d,PROPN,2d,PROPN,2d/3d,NUM
1,3d,ADJ,3d,ADJ,abstract,ADJ
2,3d,NOUN,3d,NOUN,align,NOUN
3,able,ADJ,abstract,ADJ,ambiguous,ADJ
4,abstract,ADJ,add,VERB,analysis,NOUN
5,abstract,NOUN,additional,ADJ,analyze,VERB
6,abundance,NOUN,align,NOUN,annotation,NOUN
7,accessible,ADJ,align,VERB,appeal,VERB
8,account,NOUN,ambiguous,ADJ,arbitrary,ADJ
9,accurate,ADJ,analysis,NOUN,arrangement,NOUN


In [8]:
# ── Per-topic unique stems with image counts ──────────────────────────────────
frames = []
for topic in topic_cols:
    subset = df_img[df_img['Topic'] == topic]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('AIcurated', 'AIcurated'), ('HumanCurated', 'HumanCurated')]:
        # Count images per stem
        stem_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_stems = set()
            for phrase in phrases:
                img_stems |= get_stems_from_phrase(phrase)
            for stem in img_stems:
                stem_images[stem] += 1

        sorted_items = sorted(stem_images.items())
        col_data[f'{topic}\n{label}'] = [stem_to_word.get(s, s) for s, _ in sorted_items]
        col_data[f'{topic}\n{label} Count'] = [str(c) for _, c in sorted_items]

    if col_data:
        ml = max(len(v) for v in col_data.values())
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))
    frames.append(pd.DataFrame(col_data))

    n_orig = len([x for x in col_data[f'{topic}\nOriginal'] if x])
    n_ai   = len([x for x in col_data[f'{topic}\nAIcurated'] if x])
    n_hc   = len([x for x in col_data[f'{topic}\nHumanCurated'] if x])
    print(f'{topic}: Orig={n_orig}, AI={n_ai}, HC={n_hc}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_topics = pd.concat(frames, axis=1)

csv_path = 'figures/unique_stems_by_topic.csv'
df_topics.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_topics)} rows, {len(df_topics.columns)} columns)')
df_topics.head(10)

Immediacy / Cognitive Load: Orig=196, AI=90, HC=60
Data Density / Image Clutter: Orig=166, AI=84, HC=53
Visual Encoding Clarity: Orig=227, AI=140, HC=131
Semantics / Text Legibility: Orig=144, AI=107, HC=76
Schema: Orig=166, AI=93, HC=25
Color, Symbol, and Texture Details: Orig=165, AI=119, HC=75
Aesthetics Uncertainty: Orig=70, AI=47, HC=27

Saved figures/unique_stems_by_topic.csv  (227 rows, 42 columns)


,Immediacy / Cognitive Load\nOriginal,Immediacy / Cognitive Load\nOriginal Count,Immediacy / Cognitive Load\nAIcurated,Immediacy / Cognitive Load\nAIcurated Count,Immediacy / Cognitive Load\nHumanCurated,Immediacy / Cognitive Load\nHumanCurated Count,Data Density / Image Clutter\nOriginal,Data Density / Image Clutter\nOriginal Count,Data Density / Image Clutter\nAIcurated,Data Density / Image Clutter\nAIcurated Count,...,"Color, Symbol, and Texture Details\nAIcurated","Color, Symbol, and Texture Details\nAIcurated Count","Color, Symbol, and Texture Details\nHumanCurated","Color, Symbol, and Texture Details\nHumanCurated Count",Aesthetics Uncertainty\nOriginal,Aesthetics Uncertainty\nOriginal Count,Aesthetics Uncertainty\nAIcurated,Aesthetics Uncertainty\nAIcurated Count,Aesthetics Uncertainty\nHumanCurated,Aesthetics Uncertainty\nHumanCurated Count
0,able,1,analysis,1,analysis,3,abundance,1,analyze,1,...,ambiguous,5,ambiguous,5,appeal,2,appeal,5,appeal,5
1,accessible,1,ascertain,2,analyze,5,ammount,1,area,4,...,appeal,2,appeal,2,arbitrary,1,arbitrary,1,arbitrary,1
2,accurate,1,careful,1,attention,3,analyse,1,box,2,...,arrangement,1,arrangement,85,attractive,1,attractive,1,attractive,3
3,add,1,challenge,1,clear,1,annotation,1,busy,7,...,background,4,background,5,bad,3,bad,3,clarity,5
4,allow,2,clear,4,compare,11,area,4,chart,3,...,bar,1,bar,2,black,1,blurry,1,color,8
5,analysis,2,clue,1,complicated,8,arrow,1,circle,4,...,black,9,black,14,blurry,1,bore,1,composition,1
6,analyze,2,compare,2,confuse,24,aspect,1,close,2,...,blend,1,blend,1,bore,1,calligraphy,1,confuse,19
7,ascertain,3,complex,4,datum,7,assortment,1,cluster,2,...,blue,5,blue,2,calligraphy,1,chaotic,1,contrast,8
8,assimulate,1,complicated,5,derive,2,bar,1,clutter,7,...,blur,2,blurry,3,chaotic,1,choice,1,convolute,1
9,attention,1,comprehend,6,describe,2,begin,1,color,1,...,blurry,1,bold,1,choice,3,clarity,3,distract,19


In [9]:
# ── Per-VisType unique stems with image counts ────────────────────────────────
frames = []
for vt in CANONICAL_VISTYPES:
    subset = df_img[df_img['VisType'] == vt]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('AIcurated', 'AIcurated'), ('HumanCurated', 'HumanCurated')]:
        stem_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_stems = set()
            for phrase in phrases:
                img_stems |= get_stems_from_phrase(phrase)
            for stem in img_stems:
                stem_images[stem] += 1

        sorted_items = sorted(stem_images.items())
        col_data[f'{vt}\n{label}'] = [stem_to_word.get(s, s) for s, _ in sorted_items]
        col_data[f'{vt}\n{label} Count'] = [str(c) for _, c in sorted_items]

    if col_data:
        ml = max(len(v) for v in col_data.values())
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))

    frames.append(pd.DataFrame(col_data))
    n_orig = len([x for x in col_data[f'{vt}\nOriginal'] if x])
    n_ai   = len([x for x in col_data[f'{vt}\nAIcurated'] if x])
    n_hc   = len([x for x in col_data[f'{vt}\nHumanCurated'] if x])
    print(f'{vt}: Orig={n_orig}, AI={n_ai}, HC={n_hc}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_vistype = pd.concat(frames, axis=1)

csv_path = 'figures/unique_stems_by_vistype.csv'
df_vistype.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_vistype)} rows, {len(df_vistype.columns)} columns)')
df_vistype.head(10)

Area: Orig=236, AI=185, HC=152
Bar: Orig=125, AI=111, HC=114
Cont.-ColorPatn: Orig=167, AI=134, HC=137
Glyph: Orig=186, AI=142, HC=132
Grid: Orig=187, AI=151, HC=153
Line: Orig=155, AI=125, HC=113
Node-link: Orig=217, AI=159, HC=150
Point: Orig=176, AI=143, HC=145
Text: Orig=127, AI=104, HC=109

Saved figures/unique_stems_by_vistype.csv  (236 rows, 54 columns)


,Area\nOriginal,Area\nOriginal Count,Area\nAIcurated,Area\nAIcurated Count,Area\nHumanCurated,Area\nHumanCurated Count,Bar\nOriginal,Bar\nOriginal Count,Bar\nAIcurated,Bar\nAIcurated Count,...,Point\nAIcurated,Point\nAIcurated Count,Point\nHumanCurated,Point\nHumanCurated Count,Text\nOriginal,Text\nOriginal Count,Text\nAIcurated,Text\nAIcurated Count,Text\nHumanCurated,Text\nHumanCurated Count
0,abstract,1,abstract,1,2d/3d,1,2d,1,2d,1,...,2d,1,2d/3d,2,application,1,ambiguous,1,ambiguous,1
1,accessible,1,align,1,abstract,1,align,1,ambiguous,1,...,3d,1,abstract,1,arbitrary,1,annotation,1,annotation,2
2,africa,1,ambiguous,2,align,1,analysis,1,analysis,1,...,add,1,ambiguous,1,ascertain,1,busy,1,arrangement,7
3,allow,1,analyze,1,ambiguous,2,axis,2,axis,2,...,ambiguous,1,analyze,1,big,1,calligraphy,1,attractive,1
4,ambiguous,1,appeal,1,analyze,1,bar,7,bar,7,...,animal,1,annotation,6,bit,1,chart,1,axis,6
5,analyse,1,area,2,annotation,5,black,1,black,1,...,annotation,1,arrangement,10,bunch,1,circle,1,bar,1
6,analyze,1,ascertain,1,appeal,1,blue,1,blue,1,...,appeal,1,aspect,1,busy,1,clear,4,biology,4
7,apart,1,axis,5,arrangement,16,box,1,busy,1,...,area,1,attractive,1,calligraphy,1,clutter,2,chart,3
8,appleale,1,background,1,axis,11,busy,1,change,1,...,aspect,1,axis,14,change,1,code,1,chemical,4
9,area,2,black,1,background,1,change,1,chart,3,...,axis,8,bar,1,chart,2,color,14,circle,1


## Word Reduction Pipeline: Original → HumanCurated → FinalPhrase

Using the final sub-topic phrases from `phrase_reduction_v2/`, map each image's HumanCurated phrase
to its final shortlisted phrase (if any), then extract unique stems at each level to show the
progressive word reduction: raw annotator text → human-curated phrases → final sub-topic phrases.

In [10]:
# ── Load final phrase mapping from PhraseReductionPipeline ─────────────────────
df_lineage = pd.read_csv('phrase_reduction_v2/phrase_lineage_all_vistypes.csv')

# Build unique mapping: HumanCurated phrase → FinalPhrase
# (original_phrase in lineage = HumanCurated in image_phrase_matching)
hc_to_final = {}
for _, row in df_lineage.iterrows():
    hc = row['original_phrase']
    final = row['final_subtopic_phrase']
    if isinstance(final, str) and final.strip():
        hc_to_final[hc] = final
    elif hc not in hc_to_final:
        hc_to_final[hc] = ''

# Add FinalPhrase column to df_img
df_img['FinalPhrase'] = df_img['HumanCurated'].map(hc_to_final).fillna('')

n_with_final = (df_img['FinalPhrase'] != '').sum()
n_unique_final = df_img[df_img['FinalPhrase'] != '']['FinalPhrase'].nunique()
print(f'Mapped {n_with_final}/{len(df_img)} records to a final phrase ({n_unique_final} unique final phrases)')
print(f'\nFinal phrases: {sorted(df_img[df_img["FinalPhrase"] != ""]["FinalPhrase"].unique())}')

# ── All-phrases unique stems: Original → HumanCurated → FinalPhrase ───────────
pipeline_cols = {'Original': 'Original', 'HumanCurated': 'HumanCurated', 'FinalPhrase': 'FinalPhrase'}
word_lists_pipeline = {}
for col, label in pipeline_cols.items():
    phrases = df_img[col].dropna()
    phrases = phrases[phrases.str.strip() != ''].unique()
    wp = extract_stem_pos_pairs(phrases)
    word_lists_pipeline[label] = wp
    print(f'{label}: {len(phrases)} phrases → {len(wp)} unique stems')

max_len = max(len(v) for v in word_lists_pipeline.values())
frames = []
for label in pipeline_cols.values():
    wp = word_lists_pipeline[label]
    d = pd.DataFrame(wp, columns=[f'{label}Word', f'{label}POS'])
    d = d.reindex(range(max_len)).fillna('')
    frames.append(d)

df_pipeline = pd.concat(frames, axis=1)
csv_path = 'figures/unique_stems_pipeline.csv'
df_pipeline.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({max_len} rows)')
df_pipeline.head(15)

Mapped 1678/1909 records to a final phrase (19 unique final phrases)

Final phrases: ['2D/3D', 'amount of words/context/numbers', 'color contrast/clarity', 'color variety/shading', 'distracting/confusing/unclear', 'domain-specific concepts (e.g., chemical, biology, map)', 'easy/hard to interpret', 'labels/axes/legends', 'more charts/points/lines/shapes/elements', 'much/little data/info', 'overlapping shapes/colors/lines', 'scale difference', 'shapes and lines', 'symbols', 'take longer to interpret', 'unclear colors/contrast', 'unclear meaning/confusing', 'understand/read shapes', 'word rotation/small font size']
Original: 1560 phrases → 829 unique stems
HumanCurated: 397 phrases → 363 unique stems
FinalPhrase: 19 phrases → 46 unique stems

Saved figures/unique_stems_pipeline.csv  (829 rows)


,OriginalWord,OriginalPOS,HumanCuratedWord,HumanCuratedPOS,FinalPhraseWord,FinalPhrasePOS
0,2d,PROPN,2d/3d,NUM,2d/3d,NUM
1,3d,ADJ,abstract,ADJ,axis,NOUN
2,3d,NOUN,align,NOUN,biology,NOUN
3,able,ADJ,ambiguous,ADJ,chart,NOUN
4,abstract,ADJ,analysis,NOUN,chemical,PROPN
5,abstract,NOUN,analyze,VERB,clarity,NOUN
6,abundance,NOUN,annotation,NOUN,color,NOUN
7,accessible,ADJ,appeal,VERB,color,PROPN
8,account,NOUN,arbitrary,ADJ,concept,NOUN
9,accurate,ADJ,arrangement,NOUN,confuse,VERB


In [11]:
# ── Per-topic unique stems: Original → HumanCurated → FinalPhrase ──────────────
frames = []
for topic in topic_cols:
    subset = df_img[df_img['Topic'] == topic]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('HumanCurated', 'HumanCurated'), ('FinalPhrase', 'FinalPhrase')]:
        stem_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_stems = set()
            for phrase in phrases:
                if phrase.strip() == '':
                    continue
                img_stems |= get_stems_from_phrase(phrase)
            for stem in img_stems:
                stem_images[stem] += 1

        sorted_items = sorted(stem_images.items())
        col_data[f'{topic}\n{label}'] = [stem_to_word.get(s, s) for s, _ in sorted_items]
        col_data[f'{topic}\n{label} Count'] = [str(c) for _, c in sorted_items]

    if col_data:
        ml = max(len(v) for v in col_data.values()) if any(col_data.values()) else 0
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))
    frames.append(pd.DataFrame(col_data))

    n_orig = len([x for x in col_data.get(f'{topic}\nOriginal', []) if x])
    n_hc   = len([x for x in col_data.get(f'{topic}\nHumanCurated', []) if x])
    n_fp   = len([x for x in col_data.get(f'{topic}\nFinalPhrase', []) if x])
    print(f'{topic}: Orig={n_orig}, HC={n_hc}, Final={n_fp}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_topics_pipeline = pd.concat(frames, axis=1)

csv_path = 'figures/unique_stems_by_topic_pipeline.csv'
df_topics_pipeline.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_topics_pipeline)} rows, {len(df_topics_pipeline.columns)} columns)')
df_topics_pipeline.head(10)

Immediacy / Cognitive Load: Orig=196, HC=60, Final=12
Data Density / Image Clutter: Orig=166, HC=53, Final=16
Visual Encoding Clarity: Orig=227, HC=131, Final=41
Semantics / Text Legibility: Orig=144, HC=76, Final=31
Schema: Orig=166, HC=25, Final=9
Color, Symbol, and Texture Details: Orig=165, HC=75, Final=25
Aesthetics Uncertainty: Orig=70, HC=27, Final=17

Saved figures/unique_stems_by_topic_pipeline.csv  (227 rows, 42 columns)


,Immediacy / Cognitive Load\nOriginal,Immediacy / Cognitive Load\nOriginal Count,Immediacy / Cognitive Load\nHumanCurated,Immediacy / Cognitive Load\nHumanCurated Count,Immediacy / Cognitive Load\nFinalPhrase,Immediacy / Cognitive Load\nFinalPhrase Count,Data Density / Image Clutter\nOriginal,Data Density / Image Clutter\nOriginal Count,Data Density / Image Clutter\nHumanCurated,Data Density / Image Clutter\nHumanCurated Count,...,"Color, Symbol, and Texture Details\nHumanCurated","Color, Symbol, and Texture Details\nHumanCurated Count","Color, Symbol, and Texture Details\nFinalPhrase","Color, Symbol, and Texture Details\nFinalPhrase Count",Aesthetics Uncertainty\nOriginal,Aesthetics Uncertainty\nOriginal Count,Aesthetics Uncertainty\nHumanCurated,Aesthetics Uncertainty\nHumanCurated Count,Aesthetics Uncertainty\nFinalPhrase,Aesthetics Uncertainty\nFinalPhrase Count
0,able,1,analysis,3,color,2,abundance,1,analyze,2,...,ambiguous,5,biology,2,appeal,2,appeal,5,biology,1
1,accessible,1,analyze,5,confuse,31,ammount,1,chart,78,...,appeal,2,chart,1,arbitrary,1,arbitrary,1,chemical,1
2,accurate,1,attention,3,distract,6,analyse,1,cluster,1,...,arrangement,85,chemical,2,attractive,1,attractive,3,color,8
3,add,1,clear,1,easy,184,annotation,1,clutter,18,...,background,5,clarity,50,bad,3,clarity,5,concept,1
4,allow,2,compare,11,hard,184,area,4,color,17,...,bar,2,color,162,black,1,color,8,confuse,42
5,analysis,2,complicated,8,interpret,207,arrow,1,concentrate,1,...,black,14,concept,2,blurry,1,composition,1,contrast,8
6,analyze,2,confuse,24,line,3,aspect,1,dataset,1,...,blend,1,confuse,5,bore,1,confuse,19,distract,42
7,ascertain,3,datum,7,long,38,assortment,1,datum,86,...,blue,2,contrast,50,calligraphy,1,contrast,8,domain,1
8,assimulate,1,derive,2,mean,25,bar,1,dense,24,...,blurry,3,distract,1,chaotic,1,convolute,1,easy,2
9,attention,1,describe,2,overlap,2,begin,1,detail,47,...,bold,1,domain,2,choice,3,distract,19,hard,2


In [12]:
# ── Per-VisType unique stems: Original → HumanCurated → FinalPhrase ────────────
frames = []
for vt in CANONICAL_VISTYPES:
    subset = df_img[df_img['VisType'] == vt]
    col_data = {}
    for src_col, label in [('Original', 'Original'), ('HumanCurated', 'HumanCurated'), ('FinalPhrase', 'FinalPhrase')]:
        stem_images = Counter()
        for img_name, grp in subset.groupby('ImageName'):
            phrases = grp[src_col].dropna().unique()
            img_stems = set()
            for phrase in phrases:
                if phrase.strip() == '':
                    continue
                img_stems |= get_stems_from_phrase(phrase)
            for stem in img_stems:
                stem_images[stem] += 1

        sorted_items = sorted(stem_images.items())
        col_data[f'{vt}\n{label}'] = [stem_to_word.get(s, s) for s, _ in sorted_items]
        col_data[f'{vt}\n{label} Count'] = [str(c) for _, c in sorted_items]

    if col_data:
        ml = max(len(v) for v in col_data.values()) if any(col_data.values()) else 0
        for k, v in col_data.items():
            col_data[k] = v + [''] * (ml - len(v))

    frames.append(pd.DataFrame(col_data))
    n_orig = len([x for x in col_data.get(f'{vt}\nOriginal', []) if x])
    n_hc   = len([x for x in col_data.get(f'{vt}\nHumanCurated', []) if x])
    n_fp   = len([x for x in col_data.get(f'{vt}\nFinalPhrase', []) if x])
    print(f'{vt}: Orig={n_orig}, HC={n_hc}, Final={n_fp}')

global_max = max(len(f) for f in frames)
frames = [f.reindex(range(global_max)).fillna('') for f in frames]
df_vistype_pipeline = pd.concat(frames, axis=1)

csv_path = 'figures/unique_stems_by_vistype_pipeline.csv'
df_vistype_pipeline.to_csv(csv_path, index=False)
print(f'\nSaved {csv_path}  ({len(df_vistype_pipeline)} rows, {len(df_vistype_pipeline.columns)} columns)')
df_vistype_pipeline.head(10)

Area: Orig=236, HC=152, Final=42
Bar: Orig=125, HC=114, Final=41
Cont.-ColorPatn: Orig=167, HC=137, Final=38
Glyph: Orig=186, HC=132, Final=40
Grid: Orig=187, HC=153, Final=40
Line: Orig=155, HC=113, Final=43
Node-link: Orig=217, HC=150, Final=42
Point: Orig=176, HC=145, Final=40
Text: Orig=127, HC=109, Final=43

Saved figures/unique_stems_by_vistype_pipeline.csv  (236 rows, 54 columns)


,Area\nOriginal,Area\nOriginal Count,Area\nHumanCurated,Area\nHumanCurated Count,Area\nFinalPhrase,Area\nFinalPhrase Count,Bar\nOriginal,Bar\nOriginal Count,Bar\nHumanCurated,Bar\nHumanCurated Count,...,Point\nHumanCurated,Point\nHumanCurated Count,Point\nFinalPhrase,Point\nFinalPhrase Count,Text\nOriginal,Text\nOriginal Count,Text\nHumanCurated,Text\nHumanCurated Count,Text\nFinalPhrase,Text\nFinalPhrase Count
0,abstract,1,2d/3d,1,2d/3d,8,2d,1,2d/3d,1,...,2d/3d,2,2d/3d,4,application,1,ambiguous,1,2d/3d,1
1,accessible,1,abstract,1,axis,14,align,1,ambiguous,1,...,abstract,1,axis,13,arbitrary,1,annotation,2,axis,5
2,africa,1,align,1,biology,25,analysis,1,analysis,1,...,ambiguous,1,biology,17,ascertain,1,arrangement,7,biology,13
3,allow,1,ambiguous,2,chart,10,axis,2,annotation,3,...,analyze,1,chart,17,big,1,attractive,1,chart,6
4,ambiguous,1,analyze,1,chemical,25,bar,7,arrangement,6,...,annotation,6,chemical,17,bit,1,axis,6,chemical,13
5,analyse,1,annotation,5,clarity,10,black,1,attention,1,...,arrangement,10,clarity,3,bunch,1,bar,1,clarity,4
6,analyze,1,appeal,1,color,24,blue,1,axis,9,...,aspect,1,color,17,busy,1,biology,4,color,13
7,apart,1,arrangement,16,concept,25,box,1,bar,8,...,attractive,1,concept,17,calligraphy,1,chart,3,concept,13
8,appleale,1,axis,11,confuse,13,busy,1,black,1,...,axis,14,confuse,9,change,1,chemical,4,confuse,10
9,area,2,background,1,context,6,change,1,change,1,...,bar,1,context,6,chart,2,circle,1,context,14
